# Ukrainian Folk Art Annotations: JSON to CSV (combined)

This notebook, similarly to 3.01, reads all json files from /data and saves annotations from all 4 events to a combined csv file.

## Setup
Define a common `BASE_DIR` for file paths and, when running in Google Colab, optionally mount Google Drive.

In [1]:
from pathlib import Path



IN_COLAB = False
try:
  from google.colab import drive  # type: ignore
  IN_COLAB = True
except ImportError:
  pass

if IN_COLAB:
  # Optional in Colab if you need a newer package version:
  # !pip -q install pandas

  drive.mount('/content/drive')

USE_DRIVE = IN_COLAB  # set False to use local runtime files

if USE_DRIVE:
  BASE_DIR = Path('/content/drive/MyDrive/AISTER-Crowdsourcing-Pilot/step_3')
else:
  BASE_DIR = Path('../')

In [2]:
print(f'Base directory: {BASE_DIR.resolve()}')

Base directory: C:\Users\User\Desktop\Web2Learn\AISTER-Crowdsourcing-Pilot\step_3


## Setup

Load json data from all 4 files into separate lists.

In [3]:
import json

data_dir = BASE_DIR / 'data'
json_files = sorted(data_dir.glob('*.json'))

assert len(json_files) == 5, f'Expected 5 json files in {data_dir}, found {len(json_files)}'
# Load JSON data from each file
data1, data2, data3, data4, data5 = [json.load(f.open('r', encoding='utf-8')) for f in json_files]
# Keep only the '@graph' content from each loaded JSON (ditch metadata)
graph1, graph2, graph3, graph4, graph5 = data1.get('@graph') or [], data2.get('@graph') or [], data3.get('@graph') or [], data4.get('@graph') or [], data5.get('@graph') or []

print(f'Loaded data from {len(json_files)} files:')
for i, f in enumerate(json_files, 1):
    print(f'  {i}. {f.name} with {len(graph1) if i == 1 else len(graph2) if i == 2 else len(graph3) if i == 3 else len(graph4) if i == 4 else len(graph5)} graph items')


Loaded data from 5 files:
  1. 1_ukrainian-folkart_annotations.json with 4018 graph items
  2. 2_ukrainian-folkart_annotations.json with 4018 graph items
  3. 3_ukrainian-folkart_annotations.json with 4760 graph items
  4. 4_ukrainian-folkart_annotations.json with 5829 graph items
  5. 5_ukrainian-folkart_annotations.json with 5946 graph items


## Transform annotations

Build 4 separate normalized data frames with default values where needed

In [4]:
import pandas as pd


def build_annotations_df(graph_items):
  rows = []

  for g in graph_items:
    source = (g.get('target') or {}).get('source') or ''

    rows.append({
      'created': g.get('created') or '',
      'tag': (g.get('body') or {}).get('value'),
      'europeana_id': source.split('/')[-1],
      'upvotes': (g.get('review') or {}).get('upvotes') or 0,
      'downvotes': (g.get('review') or {}).get('downvotes') or 0,
      'recommendation': (g.get('review') or {}).get('recommendation') or 'unknown',
    })

  df_local = pd.DataFrame(rows)

  created_dt = pd.to_datetime(
    df_local['created'],
    format='%a %b %d %H:%M:%S UTC %Y',
    errors='coerce',
    utc=True,
  )
  # Bring data to a suitable format
  df_local['created'] = created_dt.dt.strftime('%d.%m.%Y %H:%M:%S')
  df_local['tag'] = df_local['tag'].str.lower().str.strip()
  df_local['upvotes'] = df_local['upvotes'].astype('int')
  df_local['downvotes'] = df_local['downvotes'].astype('int')
  # Sort by europeana_id
  df_local = df_local.sort_values(by='europeana_id').reset_index(drop=True)

  return df_local

df1 = build_annotations_df(graph1)
df2 = build_annotations_df(graph2)
df3 = build_annotations_df(graph3)
df4 = build_annotations_df(graph4)
df5 = build_annotations_df(graph5)

display(df1.head())
display(df2.head())
display(df3.head())
display(df4.head())
display(df5.head())

,created,tag,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026 08:08:10,geometric patterns,HMT1,1,0,accept
1,13.02.2026 08:08:10,icon,HMT1,1,0,accept
2,13.02.2026 08:08:10,wedding couple,HMT1,1,0,accept
3,13.02.2026 08:08:10,traditional iconographic style,HMT1,0,0,unknown
4,13.02.2026 08:08:10,religious icon,HMT1,2,0,accept


,created,tag,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026 08:08:10,heart,HMT1,1,0,accept
1,13.02.2026 08:08:10,traditional iconographic style,HMT1,0,0,unknown
2,13.02.2026 08:08:10,couple,HMT1,1,0,accept
3,13.02.2026 08:08:10,decorative wooden border,HMT1,1,0,accept
4,13.02.2026 08:08:10,cross,HMT1,1,0,accept


,created,tag,europeana_id,upvotes,downvotes,recommendation
0,27.03.2026 10:42:26,blue veil,HMT1,2,0,accept
1,13.02.2026 08:08:10,wedding couple,HMT1,4,2,accept
2,13.02.2026 08:08:10,halos,HMT1,4,0,accept
3,13.02.2026 08:08:10,geometric patterns,HMT1,5,0,accept
4,13.02.2026 08:08:10,mary,HMT1,4,0,accept


,created,tag,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026 08:08:10,mary,HMT1,10,1,accept
1,13.02.2026 08:08:10,jesus,HMT1,10,0,accept
2,13.02.2026 08:08:10,heart,HMT1,11,0,accept
3,13.02.2026 08:08:10,decorative wooden border,HMT1,11,0,accept
4,13.02.2026 08:08:10,halos,HMT1,11,0,accept


,created,tag,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026 08:08:10,cross,HMT1,7,3,accept
1,29.03.2026 17:16:54,green background,HMT1,4,0,accept
2,13.02.2026 08:08:10,mary,HMT1,10,1,accept
3,13.02.2026 08:08:10,icon,HMT1,14,0,accept
4,27.03.2026 11:03:48,pair of icons,HMT1,9,0,accept


## Combine data frames

Combine all four dataframes and keep track of upvotes and downvotes created in each event for each tag.

In [5]:
combined_df = pd.concat(
    [
        df1.assign(event_number=1),
        df2.assign(event_number=2),
        df3.assign(event_number=3),
        df4.assign(event_number=4),
        df5.assign(event_number=5),
    ],
    ignore_index=True,
)

combined_df = combined_df.sort_values(['europeana_id', 'tag', 'event_number']).reset_index(drop=True)

display(combined_df.head())
print(f"Total annotations: {len(combined_df)}")
# Helper function to help us avoid saving negative values in upvotes/downvotes when calculating differences
def diff_no_negative(series):
    prev = series.shift(1).fillna(0)
    diff = series - prev
    diff = diff.where(series.ne(0), 0)
    return diff.clip(lower=0).astype(int)

combined_df['upvotes'] = combined_df.groupby(['europeana_id', 'tag'])['upvotes'].transform(diff_no_negative)
combined_df['downvotes'] = combined_df.groupby(['europeana_id', 'tag'])['downvotes'].transform(diff_no_negative)

display(combined_df.head())


,created,tag,europeana_id,upvotes,downvotes,recommendation,event_number
0,27.03.2026 10:42:26,blue veil,HMT1,2,0,accept,3
1,27.03.2026 10:42:26,blue veil,HMT1,11,0,accept,4
2,27.03.2026 10:42:26,blue veil,HMT1,11,0,accept,5
3,13.02.2026 08:08:10,couple,HMT1,1,0,accept,1
4,13.02.2026 08:08:10,couple,HMT1,1,0,accept,2


Total annotations: 24571


,created,tag,europeana_id,upvotes,downvotes,recommendation,event_number
0,27.03.2026 10:42:26,blue veil,HMT1,2,0,accept,3
1,27.03.2026 10:42:26,blue veil,HMT1,9,0,accept,4
2,27.03.2026 10:42:26,blue veil,HMT1,0,0,accept,5
3,13.02.2026 08:08:10,couple,HMT1,1,0,accept,1
4,13.02.2026 08:08:10,couple,HMT1,0,0,accept,2


## Save to csv

Save combined data frame to csv

In [6]:
combined_df.to_csv('../outputs/ukrainian-folkart-annotations.csv', index=False)